### Jira Story Details

In [3]:
# !pip install python-dotenv

# Import Required Libraries
This cell imports all necessary Python libraries and modules used throughout the notebook, such as for data processing, file handling, and API requests.

In [2]:
import requests
from requests.auth import HTTPBasicAuth
import os
from dotenv import load_dotenv

JIRA_URL = os.getenv("JIRA_URL")
JIRA_USER = os.getenv("JIRA_USER")
JIRA_API_TOKEN = os.getenv("JIRA_API_TOKEN")
STORY_KEY = os.getenv("STORY_KEY")
load_dotenv()


def fetch_jira_story_description(jira_url, user, api_token, story_key):
    url = f"{jira_url}/rest/api/3/issue/{story_key}"
    auth = HTTPBasicAuth(user, api_token)
    headers = {"Accept": "application/json"}
    response = requests.get(url, headers=headers, auth=auth)
    if response.status_code == 200:
        data = response.json()
        description = data['fields'].get('description', '')
        if isinstance(description, dict) and 'content' in description:
            # Jira Cloud returns description as Atlassian Document Format (ADF)
            def extract_text(adf):
                if isinstance(adf, dict):
                    if adf.get('type') == 'text':
                        return adf.get('text', '')
                    elif 'content' in adf:
                        return ''.join([extract_text(c) for c in adf['content']])
                elif isinstance(adf, list):
                    return ''.join([extract_text(c) for c in adf])
                return ''
            description = extract_text(description['content'])
        return description
    else:
        print("Failed to fetch story:", response.status_code, response.text)
        return ""

# Fetch description and use as context
description = fetch_jira_story_description(JIRA_URL, JIRA_USER, JIRA_API_TOKEN, STORY_KEY)
print("Story Description:", description)

MissingSchema: Invalid URL 'None/rest/api/3/issue/None': No scheme supplied. Perhaps you meant https://None/rest/api/3/issue/None?

## Fetch Jira Story Description
This cell loads environment variables and defines a function to fetch the description of a Jira story using the Jira REST API. The description is used as context for further processing.

In [ ]:
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Wrap the description in a Document expected by LangChain
story_content = [Document(page_content=description)]

# Initialize the splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

# Split the documents into chunks
chunks = splitter.split_documents(story_content)

# Print the chunks
for i, chunk in enumerate(chunks):
    # chunk is a Document, access its page_content attribute
    print(f"Chunk {i+1}:\n{chunk.page_content}\n")

## Split Jira Story Description into Chunks
This cell uses LangChain's text splitter to break the Jira story description into manageable chunks for embedding and retrieval.

In [ ]:
from langchain.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize Ollama embeddings model
embeddings = OllamaEmbeddings(model="nomic-embed-text:v1.5")

# Store the chunks in a Chroma vector database (persisted to ./test_case_db)
db = Chroma.from_documents(chunks, embeddings, persist_directory="./test_case_db")

# Create a retriever from the vector database
retriever = db.as_retriever()

## Embed and Store Chunks in Vector Database
This cell creates embeddings for the text chunks using Ollama and stores them in a Chroma vector database for efficient retrieval.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="gpt-4.1",
    temperature=0.5,
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

## Initialize LLM for BDD Test Case Generation
This cell sets up the OpenAI GPT-4 model for generating BDD test cases from the requirements.

In [ ]:
from langchain.tools import tool
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

@tool
def extract_and_generate_bdd_test_cases(query: str) -> str:
    """Extract user stories and generate BDD test cases using advanced prompt engineering and retrieval."""
    # Retrieve relevant context using retriever (RAG)
    # relevant_docs = retriever.get_relevant_documents(query)
    # context = "\n\n".join([doc.page_content for doc in relevant_docs])

    prompt_template = """
    You are an expert in Behavior-Driven Development (BDD) and software testing.
    Given the following project documentation, extract user stories and generate only two BDD test cases per User story in Gherkin format.

    ## Chain of Thought
    Let's work step by step. First, identify all user stories in the provided context, each starting with a Title. For each user story, generate two BDD (Behavior-Driven Development) test cases in Given-When-Then format. Clearly separate the test cases for each user story and include the user story title before its test cases.
    
    Context:
    {context}

    Instructions:
    1. Identify and extract user stories from the provided documentation.
    2. For each user story, create a corresponding BDD test case using the Gherkin syntax.
    3. Ensure that the generated test cases are clear, concise, and follow best practices in BDD.

    ## Few-shot Example
    Example:
    User Story: Login
    Given the user is on the login page...
    When the user enters valid credentials...
    Then the user is redirected...

    BDD Test Case:
    Feature: [Feature Name]
      Scenario: [Scenario Name]
        Given [initial context]
        When [event occurs]
        Then [ensure some outcomes]

    ## ReAct
    Thought: What user stories are present in the context?
    Action: List all user story titles.
    Observation: Titles found.
    Thought: For each title, generate two BDD test cases.
    Action: Write BDD test cases in Given-When-Then format.
    Observation: Test cases written.
    Final Answer: Present all user stories and their BDD test cases.
    """
    relevant_docs = retriever.get_relevant_documents(query)
    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    prompt = PromptTemplate(template=prompt_template, input_variables=['context'])
    formatted_prompt = prompt.format(context=context)

    qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
    return qa_chain.invoke({"query": formatted_prompt})

## Tool for Extracting and Generating BDD Test Cases
This cell defines a LangChain tool that uses prompt engineering and retrieval to extract user stories and generate BDD test cases in Gherkin format.

In [ ]:
from langgraph.prebuilt import create_react_agent

qa_agent = create_react_agent(
    model=llm,
    tools=[extract_and_generate_bdd_test_cases],
    prompt=(
    "Generate BDD test cases from the requirements using the provided context and PMRs.Please write only one gerkin test which covers complete user story to perform testing."
    ),
    name="qa_agent",
)

## Create ReAct Agent for BDD Test Case Generation
This cell creates a ReAct agent using LangGraph to generate BDD test cases from requirements and context.

In [ ]:
from langchain_core.messages import convert_to_messages


def pretty_print_message(message, indent=False):
    pretty_message = message.pretty_repr(html=True)
    if not indent:
        print(pretty_message)
        return

    indented = "\n".join("\t" + c for c in pretty_message.split("\n"))
    print(indented)


def pretty_print_messages(update, last_message=False):
    is_subgraph = False
    if isinstance(update, tuple):
        ns, update = update
        # skip parent graph updates in the printouts
        if len(ns) == 0:
            return

        graph_id = ns[-1].split(":")[0]
        print(f"Update from subgraph {graph_id}:")
        print("\n")
        is_subgraph = True

    for node_name, node_update in update.items():
        update_label = f"Update from node {node_name}:"
        if is_subgraph:
            update_label = "\t" + update_label

        print(update_label)
        print("\n")

        messages = convert_to_messages(node_update["messages"])
        if last_message:
            messages = messages[-1:]

        for m in messages:
            pretty_print_message(m, indent=is_subgraph)
        print("\n")

## Pretty-Print Agent Messages
This cell defines utility functions to pretty-print messages and outputs from the agents for better readability.

In [ ]:
for chunk in qa_agent.stream(
    {"messages": [{"role": "user", "content": "Generate BDD test cases from the requirements using the provided context and PMRs. Please write only one gerkin test which covers complete user story to perform testing."}]}
):
    pretty_print_messages(chunk)

## Stream BDD Test Case Generation Results
This cell streams the output of the BDD test case generation agent and displays the results using the pretty-print functions.

In [ ]:
from langchain.tools import tool
import requests

SERVER_URL = "http://localhost:3000"

@tool
def open_browser(_=None) -> str:
    """Open Chrome browser using WebdriverIO server."""
    try:
        resp = requests.post(f"{SERVER_URL}/open")
        return resp.text
    except Exception as e:
        return f"Error calling WebdriverIO server: {e}"

@tool
def navigate_to_url(url: str) -> str:
    """Navigate to the given URL using WebdriverIO server."""
    try:
        resp = requests.post(f"{SERVER_URL}/navigate", json={"url": url})
        return resp.text
    except Exception as e:
        return f"Error calling WebdriverIO server: {e}"

@tool
def click_action(element_selector: str) -> str:
    """Click an element in Chrome browser using WebdriverIO server."""
    try:
        resp = requests.post(f"{SERVER_URL}/click", json={"selector": element_selector})
        return resp.text
    except Exception as e:
        return f"Error calling WebdriverIO server: {e}"

@tool
def set_value_action(element_selector: str, value: str) -> str:
    """Set value for an element in Chrome browser using WebdriverIO server."""
    try:
        resp = requests.post(f"{SERVER_URL}/set-value", json={"selector": element_selector, "value": value})
        return resp.text
    except Exception as e:
        return f"Error calling WebdriverIO server: {e}"

@tool
def get_page_source() -> str:
    """Get the HTML source of the current page using WebdriverIO server."""
    try:
        resp = requests.post(f"{SERVER_URL}/source")
        return resp.text
    except Exception as e:
        return f"Error calling WebdriverIO server: {e}"

@tool
def wait_for_displayed(element_selector: str, timeout: int = 5000) -> str:
    """Wait for an element to be displayed within the given timeout (ms)."""
    try:
        resp = requests.post(f"{SERVER_URL}/wait-for-displayed", json={"selector": element_selector, "timeout": timeout})
        return resp.text
    except Exception as e:
        return f"Error calling WebdriverIO server: {e}"

@tool
def wait_for_enabled(element_selector: str, timeout: int = 5000) -> str:
    """Wait for an element to be enabled within the given timeout (ms)."""
    try:
        resp = requests.post(f"{SERVER_URL}/wait-for-enabled", json={"selector": element_selector, "timeout": timeout})
        return resp.text
    except Exception as e:
        return f"Error calling WebdriverIO server: {e}"

@tool
def scroll_to_element(element_selector: str) -> str:
    """Scroll to the specified element in the browser."""
    try:
        resp = requests.post(f"{SERVER_URL}/scroll-to", json={"selector": element_selector})
        return resp.text
    except Exception as e:
        return f"Error calling WebdriverIO server: {e}"

@tool
def get_text(element_selector: str) -> str:
    """Get the text content of the specified element."""
    try:
        resp = requests.post(f"{SERVER_URL}/get-text", json={"selector": element_selector})
        return resp.text
    except Exception as e:
        return f"Error calling WebdriverIO server: {e}"
    
@tool
def reset_browser() -> str:
    """Reset the browser session (close and reopen for a fresh start)."""
    try:
        resp = requests.post(f"{SERVER_URL}/reset")
        return resp.text
    except Exception as e:
        return f"Error calling WebdriverIO server: {e}"

@tool
def close_browser() -> str:
    """Close the current browser session."""
    try:
        resp = requests.post(f"{SERVER_URL}/close")
        return resp.text
    except Exception as e:
        return f"Error calling WebdriverIO server: {e}"

## Define WebdriverIO Browser Automation Tools
This cell defines LangChain tools for browser automation using a local WebdriverIO server, enabling browser actions like open, navigate, click, set value, and more.

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.agents import initialize_agent, AgentType
import pandas as pd


qa_automation_agent = create_react_agent(
    model=llm,
    tools=[open_browser, navigate_to_url, click_action, set_value_action, get_page_source, reset_browser, close_browser],
    prompt=(
        """
        You are a QA Test Automation Engineer.
        Your task is to execute and validate web application scenarios provided by the qa_agent, following each BDD test case step by step.
        For each scenario:
        - Use all available tools to interact with the web application as required.
        - Carefully follow the steps in the BDD scenario, performing actions and verifying expected outcomes.
        - After each step, check for correctness and capture any discrepancies or issues.
        - Provide a detailed report of actions taken, results observed, and any bugs or failures encountered.
        - If a step cannot be completed, explain why and suggest possible causes or next steps.
        Ensure your testing is thorough, methodical, and clearly documented.
        """
    ),
    name="qa_automation_agent",
)


## Create QA Automation Agent for Web Testing
This cell creates a QA automation agent using LangGraph and the defined browser tools to perform web testing tasks based on scenarios.

In [ ]:
for chunk in qa_automation_agent.stream(
    {"messages": [{"role": "user", "content": "Perform Testing for the scenario provided from the qa_agent"}]}
):
    pretty_print_messages(chunk)

## Stream QA Automation Agent Results
This cell streams the output of the QA automation agent as it performs web testing steps and displays the results.

In [ ]:
# !pip install langgraph_supervisor

## Install langgraph_supervisor Package
This cell installs the langgraph_supervisor package, which is required for managing multiple agents.

In [ ]:
from langgraph_supervisor import create_supervisor
from langchain.chat_models import init_chat_model

qa_manager_agent = create_supervisor(
    model=llm,
    agents=[qa_agent, qa_automation_agent],
    prompt=(
        "You are a qa  manager managing two qa agents:\n"
        "- a qa_agent. Assign BDD test case generation - related tasks to this agent\n"
        "- a qa_automation_agent. Assign QA test automation tasks to this agent\n"
        "Assign work to one agent at a time, do not call agents in parallel.\n"
        "Do not do any work yourself."
    ),
    add_handoff_back_messages=True,
    output_mode="full_history",
).compile()

## Create QA Manager Agent to Orchestrate Agents
This cell creates a QA manager agent using langgraph_supervisor to coordinate the BDD and QA automation agents, assigning tasks as needed.

In [ ]:
from IPython.display import display, Image

display(Image(qa_manager_agent.get_graph().draw_mermaid_png()))

## Visualize Agent Workflow Graph
This cell visualizes the workflow graph of the QA manager agent, showing the relationships and task assignments between agents.

In [ ]:
for chunk in qa_manager_agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "qa_agent:\n"
                    "1. Search for the most recent 'in progress' user stories in the TAP project.\n"
                    "2. For one selected story, design a robust BDD test scenario suitable for smoke testing. "
                    "Ensure the scenario is clear, stable, and follows Gherkin syntax.\n\n"
                    "qa_automation_agent:\n"
                    "Once the BDD test case is ready, automate the scenario using the available browser automation tools. "
                    "Execute the test step by step, validate each outcome, and provide a detailed report of the results, including any issues or failures encountered."
                ),
            }
        ]
    },
):
    pretty_print_messages(chunk, last_message=True)

final_message_history = chunk["supervisor"]["messages"]